# Uniform three-seed ensemble evaluation (Kaggle, inference only)

Compare each of three fixed Stage 4 checkpoints (seeds 17, 43, 73) with their **predeclared equal-weight RGB output average**. O-HAZE uses all 45 paired images as the primary evaluation. The optional Dense-Haze set uses its pooled 55-pair release as secondary context, not as an official challenge-only result.

No training, test-set coefficient selection, or per-image model selection occurs. Ensemble weights are fixed at 1/3 before scoring. Report evaluator-compatible PSNR, SSIM, MS-SSIM, MAE, and mean CIEDE2000; CIEDE2000 is marked unavailable if `scikit-image` is absent. Paired bootstrap resamples images and gives intervals conditional on these fixed datasets and checkpoints.

Planning estimate: about **5–15 minutes for O-HAZE** and **10–25 minutes if Dense-Haze is also run** on Kaggle 2×T4. Model inference is sequential and likely uses one T4; do not assume two GPUs halve runtime. Actual timing depends on Kaggle allocation, TensorFlow startup, checkpoint loading, and contention. The scoring cell reports its elapsed time.

Run cells in order. Edit the next cell's paths if your Kaggle attachments differ. The final cell creates and verifies a ZIP containing all files from this run.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import time
import zipfile

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
DENSE_ROOT = Path("/kaggle/input/datasets/lucky02655/den-haze-test")
IMAGE_SIZE = 512
SEEDS = (17, 43, 73)
ENSEMBLE_WEIGHTS = {seed: 1.0 / len(SEEDS) for seed in SEEDS}
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
METRICS = ("psnr", "ssim", "ms_ssim", "mae", "deltaE2000")

RUN_ID = time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1_000_000_000:09d}"
OUTPUT_ROOT = Path("/kaggle/working/three_seed_uniform_ensemble") / f"run_{RUN_ID}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

assert PACKAGE.is_dir(), f"Missing reviewer package: {PACKAGE}"
assert (OHAZE_ROOT / "hazy").is_dir(), f"Missing O-HAZE hazy directory: {OHAZE_ROOT / 'hazy'}"
assert (OHAZE_ROOT / "GT").is_dir(), f"Missing O-HAZE GT directory: {OHAZE_ROOT / 'GT'}"
SCRIPT_ROOT = PACKAGE / "code"
sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, load_image_pair
from eca_ldnet.evaluate import predict_full
from eca_ldnet.model import load_pretrained

CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in SEEDS
}
missing = [str(path) for path in CHECKPOINTS.values() if not path.is_file()]
assert not missing, f"Missing fixed Stage 4 checkpoint(s): {missing}"

print("TensorFlow:", tf.__version__)
print("Visible GPUs:", [device.name for device in tf.config.list_physical_devices("GPU")])
print("Run output directory:", OUTPUT_ROOT)
print("Fixed ensemble weights:", ENSEMBLE_WEIGHTS)

In [ ]:
def collect_pairs(root, expected_count=None):
    hazy_dir, gt_dir = root / "hazy", root / "GT"
    if not hazy_dir.is_dir() or not gt_dir.is_dir():
        return [], {"status": "skipped", "reason": "hazy or GT directory missing", "root": str(root)}
    hazy_files = sorted(path for path in hazy_dir.iterdir() if path.suffix.lower() in IMAGE_EXTS)
    pairs, unmatched = [], []
    for hazy_path in hazy_files:
        gt_path = get_gt_path(hazy_path.name, str(gt_dir))
        if gt_path is None:
            unmatched.append(hazy_path.name)
        else:
            pairs.append({"image": hazy_path.name, "hazy_path": str(hazy_path), "gt_path": str(gt_path)})
    info = {"status": "available", "root": str(root), "hazy_files": len(hazy_files),
            "matched_pairs": len(pairs), "unmatched_hazy": unmatched}
    if expected_count is not None and len(pairs) != expected_count:
        info.update({"status": "skipped", "reason": f"expected {expected_count} pairs, found {len(pairs)}"})
        return [], info
    if unmatched:
        info.update({"status": "skipped", "reason": "one or more hazy images lack ground truth"})
        return [], info
    return pairs, info


def load_pairs(pairs, dataset_name, required=False):
    loaded, unreadable = [], []
    for pair in pairs:
        hazy, clean = load_image_pair(
            pair["hazy_path"], pair["gt_path"], size=IMAGE_SIZE, preprocessing="radiometric"
        )
        if hazy is None or clean is None or hazy.shape != clean.shape:
            unreadable.append(pair["image"])
        else:
            loaded.append({"image": pair["image"], "hazy": hazy, "clean": clean,
                           "hazy_path": pair["hazy_path"], "gt_path": pair["gt_path"]})
    if unreadable and required:
        raise RuntimeError(f"Unreadable/mismatched {dataset_name} pair(s): {unreadable}")
    return loaded, unreadable

ohaze_pairs, ohaze_info = collect_pairs(OHAZE_ROOT, expected_count=45)
assert ohaze_info["status"] == "available", f"O-HAZE must contain all 45 pairs: {ohaze_info}"
dense_pairs, dense_info = collect_pairs(DENSE_ROOT, expected_count=55)
if dense_info["status"] == "skipped":
    print("Dense-Haze skipped:", dense_info["reason"])

PAIR_SETS = {"O-HAZE": ohaze_pairs}
if dense_info["status"] == "available":
    PAIR_SETS["Dense-Haze pooled release (55)"] = dense_pairs
DATASETS, UNREADABLE = {}, {}
for dataset_name, pairs in PAIR_SETS.items():
    loaded, unreadable = load_pairs(pairs, dataset_name, required=(dataset_name == "O-HAZE"))
    if unreadable and dataset_name != "O-HAZE":
        dense_info.update({"status": "skipped", "reason": f"unreadable/mismatched pairs: {unreadable}"})
        print("Dense-Haze skipped:", dense_info["reason"])
        continue
    if dataset_name == "O-HAZE":
        assert len(loaded) == 45, f"Expected 45 readable O-HAZE pairs, got {len(loaded)}"
    DATASETS[dataset_name] = loaded
    UNREADABLE[dataset_name] = unreadable
    print(f"{dataset_name}: {len(loaded)} readable pairs")

In [ ]:
try:
    from skimage.color import deltaE_ciede2000, rgb2lab
    HAS_DELTA_E = True
except ImportError:
    HAS_DELTA_E = False
    print("scikit-image missing: CIEDE2000 will be NaN; other metrics remain available.")


def score_prediction(prediction, clean):
    pred = np.clip(np.asarray(prediction, dtype=np.float32), 0.0, 1.0)
    gt = np.clip(np.asarray(clean, dtype=np.float32), 0.0, 1.0)
    diff = pred.astype(np.float64) - gt.astype(np.float64)
    scores = {
        "psnr": float(tf.image.psnr(pred[None], gt[None], 1.0).numpy().reshape(-1)[0]),
        "ssim": float(tf.image.ssim(pred[None], gt[None], 1.0).numpy().reshape(-1)[0]),
        "mae": float(np.mean(np.abs(diff))),
    }
    try:
        scores["ms_ssim"] = float(tf.image.ssim_multiscale(pred[None], gt[None], 1.0).numpy().reshape(-1)[0])
    except (tf.errors.InvalidArgumentError, ValueError):
        scores["ms_ssim"] = float("nan")
    scores["deltaE2000"] = float(np.mean(deltaE_ciede2000(rgb2lab(pred), rgb2lab(gt)))) if HAS_DELTA_E else float("nan")
    return scores

print("CIEDE2000 available:", HAS_DELTA_E)

## Score fixed checkpoints and equal-weight ensemble

For each image, run each checkpoint once. Average the three clipped RGB predictions with fixed weights `(1/3, 1/3, 1/3)`, then score the ensemble and each seed on the same ground truth. Checkpoint outputs and metric rows are not used to change ensemble weights.

In [ ]:
score_rows = []
checkpoint_hashes = {}
scoring_started = time.perf_counter()

for dataset_name, records in DATASETS.items():
    print(f"Scoring {dataset_name}: {len(records)} images")
    seed_predictions = {seed: [] for seed in SEEDS}
    for seed, checkpoint in CHECKPOINTS.items():
        digest = hashlib.sha256()
        with checkpoint.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        checkpoint_hashes[seed] = {"path": str(checkpoint), "sha256": digest.hexdigest()}

        print(f"  Loading fixed checkpoint, seed {seed}")
        model = load_pretrained(str(checkpoint), compile=False)
        shape = model.input_shape
        if shape[1] is not None and shape[1] != IMAGE_SIZE:
            raise ValueError(f"Seed {seed} expects height {shape[1]}, not {IMAGE_SIZE}")
        if shape[2] is not None and shape[2] != IMAGE_SIZE:
            raise ValueError(f"Seed {seed} expects width {shape[2]}, not {IMAGE_SIZE}")

        for index, record in enumerate(records, start=1):
            pred = predict_full(model, record["hazy"])
            seed_predictions[seed].append(pred)
            score_rows.append({"dataset": dataset_name, "image": record["image"],
                               "method": f"seed_{seed}", "seed": seed,
                               **score_prediction(pred, record["clean"])})
            if index % 10 == 0 or index == len(records):
                print(f"    seed {seed}: {index}/{len(records)}")
        del model
        tf.keras.backend.clear_session()

    for index, record in enumerate(records):
        stacked = np.stack([seed_predictions[seed][index] for seed in SEEDS], axis=0)
        ensemble = np.tensordot(
            np.asarray([ENSEMBLE_WEIGHTS[seed] for seed in SEEDS], dtype=np.float32),
            stacked, axes=(0, 0)
        )
        ensemble = np.clip(ensemble, 0.0, 1.0).astype(np.float32)
        score_rows.append({"dataset": dataset_name, "image": record["image"],
                           "method": "uniform_ensemble", "seed": "all",
                           **score_prediction(ensemble, record["clean"])})
    del seed_predictions

scores = pd.DataFrame(score_rows)
assert not scores.empty
expected_rows = sum(len(records) for records in DATASETS.values()) * (len(SEEDS) + 1)
assert len(scores) == expected_rows, f"Expected {expected_rows} metric rows; got {len(scores)}"
scores.to_csv(OUTPUT_ROOT / "per_image_scores.csv", index=False)
print(f"Saved {len(scores):,} image-method rows to per_image_scores.csv")
print(f"Scoring-cell elapsed time: {(time.perf_counter() - scoring_started) / 60:.1f} minutes")

In [ ]:
# Descriptive method summaries and paired ensemble-minus-seed differences.
summary_rows = []
for (dataset, method), group in scores.groupby(["dataset", "method"], sort=False):
    for metric in METRICS:
        values = group[metric].to_numpy(dtype=float)
        finite = values[np.isfinite(values)]
        summary_rows.append({"dataset": dataset, "method": method, "metric": metric,
                             "n_images": int(len(finite)),
                             "mean": float(np.mean(finite)) if len(finite) else np.nan,
                             "std": float(np.std(finite, ddof=1)) if len(finite) > 1 else np.nan})
method_summary = pd.DataFrame(summary_rows)
method_summary.to_csv(OUTPUT_ROOT / "dataset_method_summary.csv", index=False)

paired_rows = []
for dataset in DATASETS:
    ensemble_rows = scores[(scores["dataset"] == dataset) & (scores["method"] == "uniform_ensemble")].set_index("image")
    for seed in SEEDS:
        seed_rows = scores[(scores["dataset"] == dataset) & (scores["method"] == f"seed_{seed}")].set_index("image")
        if set(ensemble_rows.index) != set(seed_rows.index):
            raise RuntimeError(f"Image pairing mismatch for {dataset}, seed {seed}")
        for image in ensemble_rows.index:
            ens, single = ensemble_rows.loc[image], seed_rows.loc[image]
            row = {"dataset": dataset, "image": image, "seed": seed}
            for metric in METRICS:
                row[f"ensemble_{metric}"] = ens[metric]
                row[f"seed_{metric}"] = single[metric]
                row[f"delta_{metric}"] = ens[metric] - single[metric]
            paired_rows.append(row)
paired = pd.DataFrame(paired_rows)
paired.to_csv(OUTPUT_ROOT / "paired_ensemble_vs_seed_per_image.csv", index=False)

DIRECTION = {"psnr": "higher", "ssim": "higher", "ms_ssim": "higher",
             "mae": "lower", "deltaE2000": "lower"}
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []
for (dataset, seed), comparison in paired.groupby(["dataset", "seed"], sort=False):
    for metric in METRICS:
        values = comparison[f"delta_{metric}"].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if not len(values):
            continue
        draws = rng.integers(0, len(values), size=(BOOTSTRAP_SAMPLES, len(values)))
        boot_means = values[draws].mean(axis=1)
        better = values > 0 if DIRECTION[metric] == "higher" else values < 0
        bootstrap_rows.append({
            "dataset": dataset, "ensemble_vs_seed": int(seed), "metric": metric,
            "n_images": int(len(values)), "mean_delta_ensemble_minus_seed": float(np.mean(values)),
            "bootstrap_ci95_low": float(np.quantile(boot_means, 0.025)),
            "bootstrap_ci95_high": float(np.quantile(boot_means, 0.975)),
            "win_fraction_images": float(np.mean(better)),
            "direction_for_win": DIRECTION[metric],
            "bootstrap_unit": "paired image; conditional on fixed checkpoints",
        })
bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(OUTPUT_ROOT / "paired_ensemble_bootstrap_summary.csv", index=False)
print("PSNR summary:")
print(method_summary[method_summary["metric"] == "psnr"].to_string(index=False))
print("Ensemble-minus-seed PSNR bootstrap:")
print(bootstrap_summary[bootstrap_summary["metric"] == "psnr"].to_string(index=False))

In [ ]:
# Figures: absolute mean PSNR and paired ensemble-minus-seed PSNR with image-bootstrap intervals.
method_order = [f"seed_{seed}" for seed in SEEDS] + ["uniform_ensemble"]
colors = ["#4c78a8", "#72a0c1", "#9dc2d9", "#d17a22"]
datasets_present = list(DATASETS)
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = method_summary[(method_summary["dataset"] == dataset) & (method_summary["metric"] == "psnr")]
    part = part.set_index("method").reindex(method_order).dropna(subset=["mean"])
    ax.bar(part.index, part["mean"], color=colors[:len(part)])
    ax.set_title(dataset)
    ax.set_ylabel("Mean PSNR (dB)")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "mean_psnr_by_seed_and_ensemble.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

psnr_boot = bootstrap_summary[bootstrap_summary["metric"] == "psnr"]
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6 * len(datasets_present), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets_present):
    part = psnr_boot[psnr_boot["dataset"] == dataset].sort_values("ensemble_vs_seed")
    x = np.arange(len(part))
    mean = part["mean_delta_ensemble_minus_seed"].to_numpy()
    low = part["bootstrap_ci95_low"].to_numpy()
    high = part["bootstrap_ci95_high"].to_numpy()
    ax.errorbar(x, mean, yerr=np.vstack([mean - low, high - mean]), fmt="o", capsize=5, color="#355c7d")
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_xticks(x, [f"seed {seed}" for seed in part["ensemble_vs_seed"]])
    ax.set_ylabel("Ensemble − seed PSNR (dB)")
    ax.set_title(dataset)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "paired_ensemble_psnr_differences.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
dataset_manifest = {
    "O-HAZE": {**ohaze_info, "readable_pairs": len(DATASETS["O-HAZE"]),
               "images": [row["image"] for row in DATASETS["O-HAZE"]]},
    "Dense-Haze pooled release": dense_info,
}
if "Dense-Haze pooled release (55)" in DATASETS:
    dataset_manifest["Dense-Haze pooled release"].update({
        "readable_pairs": len(DATASETS["Dense-Haze pooled release (55)" ]),
        "images": [row["image"] for row in DATASETS["Dense-Haze pooled release (55)" ]],
        "interpretation": "pooled 55-pair release; not official challenge-only evaluation",
    })
manifest = {
    "experiment": "fixed three-seed uniform output ensemble",
    "scope": "inference-only descriptive comparison; no training or evaluation-set weight selection",
    "resolution": [IMAGE_SIZE, IMAGE_SIZE],
    "color_order": "RGB",
    "input_preprocessing": "radiometric [0,1], evaluator load_image_pair convention",
    "ensemble": {"rule": "arithmetic mean of clipped RGB checkpoint outputs",
                 "weights_by_seed": ENSEMBLE_WEIGHTS},
    "checkpoints": checkpoint_hashes,
    "metrics": list(METRICS),
    "metric_conventions": {
        "psnr_ssim_ms_ssim": "TensorFlow tf.image, max_val=1.0",
        "mae": "mean absolute error over all RGB values",
        "deltaE2000": "mean CIEDE2000 after skimage rgb2lab on clipped RGB [0,1]",
    },
    "datasets": dataset_manifest,
    "bootstrap": {"samples": BOOTSTRAP_SAMPLES, "seed": BOOTSTRAP_SEED,
                   "unit": "paired image", "interpretation": "conditional on fixed datasets and checkpoints"},
    "limitations": [
        "ensemble result is a fixed-weight inference comparison, not a new trained method",
        "does not establish state-of-the-art performance or deployment readiness",
        "Dense-Haze is pooled-release context and remains secondary",
        "inference loop is sequential; no multi-GPU distribution strategy is used",
    ],
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "opencv": cv2.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with (OUTPUT_ROOT / "run_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, indent=2)
print("Run outputs:", sorted(path.name for path in OUTPUT_ROOT.iterdir()))

## Package this run's outputs

Run this as the final cell after scoring, summaries, plots, and manifest generation. Each notebook run uses a fresh output folder, so the archive contains only files generated by that run.

In [ ]:
ZIP_PATH = Path("/kaggle/working/three_seed_uniform_ensemble.zip")
output_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
if not output_files:
    raise RuntimeError(f"No output files found under {OUTPUT_ROOT}; run analysis cells first.")
expected_members = {path.relative_to(OUTPUT_ROOT).as_posix() for path in output_files}
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_files:
        archive.write(path, arcname=path.relative_to(OUTPUT_ROOT).as_posix())
with zipfile.ZipFile(ZIP_PATH, mode="r") as archive:
    archived_members = set(archive.namelist())
    damaged_member = archive.testzip()
if archived_members != expected_members:
    raise RuntimeError(f"ZIP member mismatch; missing={sorted(expected_members - archived_members)}, unexpected={sorted(archived_members - expected_members)}")
if damaged_member is not None:
    raise RuntimeError(f"ZIP integrity check failed at {damaged_member}")
print(f"Verified {len(expected_members)} output files in {ZIP_PATH}")
print(f"Archive size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MiB")
for member in sorted(archived_members):
    print(" -", member)